# Experiment 4 — Multi-Agent vs Single-Agent
## Context Engineering: Compression by Architecture

**The problem:** one agent doing a multi-part research task must load every paper into
one window. With 4 papers × ~15k tokens each, context grows to ~60–80k tokens — all
competing for the model's attention at once.

**The fix:** delegate each area to a *specialist sub-agent* that works in its own
isolated window and returns only a short summary. The orchestrator sees only 3
summaries (~1–2k tokens), never the raw papers.

> "Context compression by architecture" — the boundary between agents IS the
> compression mechanism. (Research doc, Section 8.3)

**How parallel execution works here:**
1. Orchestrator (Sonnet 4.6) receives the task and sees 3 independent tool descriptions
2. It returns **all 3 tool calls in ONE response turn** — recognised as independent
3. Strands' built-in `ConcurrentToolExecutor` fires all 3 as asyncio tasks **simultaneously**
4. Each sub-agent runs in its own isolated context window
5. When all 3 finish, their summaries enter the orchestrator's window
6. Orchestrator writes the final comparison from the summaries alone


In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from strands import Agent, tool
from strands.models import BedrockModel
from core.settings import settings
from tools.load_document import load_document
from agents.subagents import peak_input_tokens

SONNET_MODEL = "us.anthropic.claude-sonnet-4-6"  # orchestrator + sub-agents
HAIKU_MODEL  = settings.bedrock_model_id           # single-agent arm (baseline)

TASK = (
    "Compare how three areas handle long-context problems: "
    "(1) attention/architecture, (2) position effects, (3) retrieval. "
    "Cover the relevant paper(s) for each area."
)
SINGLE_PAPERS = ["1706.03762", "2307.03172", "2404.06654", "2005.11401"]

print(f"Haiku  (single-agent arm): {HAIKU_MODEL}")
print(f"Sonnet (multi-agent arm):  {SONNET_MODEL}")

Haiku  (single-agent arm): us.anthropic.claude-3-5-haiku-20241022-v1:0
Sonnet (multi-agent arm):  us.anthropic.claude-sonnet-4-6


In [2]:
def _build_model(model_id: str, max_tokens: int | None = None) -> BedrockModel:
    return BedrockModel(
        model_id=model_id,
        region_name=settings.aws_region,
        temperature=settings.temperature,
        max_tokens=max_tokens or settings.output_max_tokens,
    )


def _result_text(result) -> str:
    msg = getattr(result, "message", None)
    if isinstance(msg, dict):
        return "".join(b.get("text", "") for b in msg.get("content", []) if "text" in b)
    return str(result)


print("Helpers defined.")


Helpers defined.


---
## Arm 1 — Single Agent (Load Everything)

One agent, one window. It must load all 4 papers to answer the 3-part question.
Every paper enters its context simultaneously — this is the expensive, bloated path.


In [3]:
single_system = (
    "You are a research analyst. Use load_document to read ALL of these papers: "
    f"{SINGLE_PAPERS}. Then write a structured comparison of how three areas "
    "handle long-context problems: (1) attention/architecture, (2) position effects, "
    "(3) retrieval. Cite paper ids."
)

single_agent = Agent(
    model=_build_model(HAIKU_MODEL),
    tools=[load_document],
    system_prompt=single_system,
    callback_handler=None,
)

t0 = time.time()
single_result = single_agent(TASK)
single_latency = time.time() - t0

single_peak   = peak_input_tokens(single_result)
single_answer = _result_text(single_result)
print("Single agent done.")


Single agent done.


In [ ]:
print(f"Peak context tokens: {single_peak:,}")
print(f"Latency:             {single_latency:.1f}s")
print()
print("Answer (first 1000 chars):")
print(single_answer[:1000])


Peak context tokens: 80,059
Latency:             38.4s

Answer (first 400 chars):
Based on the analysis of these papers, here's a structured comparison of how three areas handle long-context problems:

1. Attention/Architecture
- Transformer (1706.03762): 
  - Introduced self-attention mechanism that allows direct connections between all positions in a sequence
  - Multi-head attention enables attending to different representation subspaces
  - Constant number of operations between any two positions, unlike recurrent or convolutional approaches

- Lost in the Middle (2307.03172):
  - Reveals that despite self-attention's theoretical capability, models struggle to effectively use information across long contexts
  - Performance degrades significantly when relevant information is placed in the middle of long contexts
  - Models show a U-shaped performance curve, performing best with information at the beginning or end

- RULER (2404.06654):
  - Demonstrates that extended context length d

---
## Arm 2 — Multi-Agent (Real Agents-as-Tools)

Each specialist sees **only its own papers** in **its own isolated window**.
The orchestrator sees **only the 3 short summaries** — never the raw papers.

```
Orchestrator context window        Sub-agent windows (never seen by orchestrator)
─────────────────────────────      ──────────────────────────────────────────────
system prompt (~200 tokens)        attention specialist: paper 1706.03762 (~15k tok)
task (~50 tokens)                  position specialist:  papers 2307/2404 (~30k tok)
summary A (~300 tokens)            retrieval specialist: paper 2005.11401 (~20k tok)
summary B (~300 tokens)
summary C (~300 tokens)
─────────────────────────
≈ 1–2k tokens total                ≈ 15–30k tokens each (hidden from parent)
```


### Step 1 — Specialist @tools (each wraps a fresh sub-agent)

Each `@tool` function:
1. Creates a **fresh** `Agent` — its own BedrockModel, its own context window
2. Calls `load_document` to read only its assigned papers
3. Returns **only the summary string** — the raw paper work is isolated inside

`callback_handler=None` silences streaming; `subagent_summary_tokens` caps output at
~1200 tokens so each summary stays concise (~300 words).

In [5]:
import threading

_subagent_usage: list[dict] = []  # out-of-band capture — the orchestrator never sees this


def _run_specialist(name: str, area: str, papers: list[str], task: str) -> str:
    """Run an isolated specialist sub-agent; record its usage + timing out-of-band."""
    system_prompt = (
        f"You are a research specialist on {area}. "
        f"Use load_document to read ONLY these papers: {papers}. "
        "Write a focused summary of how this area handles long-context problems, "
        "in 300 words or fewer, citing the paper id(s). Output only the summary."
    )
    agent = Agent(
        model=_build_model(SONNET_MODEL, max_tokens=settings.subagent_summary_tokens),
        tools=[load_document],
        system_prompt=system_prompt,
        callback_handler=None,
    )
    t_start = time.time()
    result = agent(f"{task}\n\nYour area: {area}. Papers: {papers}.")
    t_end = time.time()

    _subagent_usage.append({
        "name": name,
        "area": area,
        "papers": papers,
        "peak_tokens": peak_input_tokens(result),
        "t_start": t_start,
        "t_end":   t_end,
        "thread":  threading.current_thread().name,
    })
    return _result_text(result)


@tool
def research_attention(task: str) -> str:
    """Summarize how attention / transformer architecture handles long context.

    Delegates to a specialist sub-agent that reads the attention paper(s) and
    returns a concise (~300 word) summary.

    Args:
        task: The overall comparison task, for context.
    """
    return _run_specialist("attention", "attention/transformer architecture",
                           ["1706.03762"], task)


@tool
def research_position(task: str) -> str:
    """Summarize how input POSITION affects long-context performance.

    Delegates to a specialist sub-agent that reads the position-effect papers and
    returns a concise (~300 word) summary.

    Args:
        task: The overall comparison task, for context.
    """
    return _run_specialist("position", "position effects in long context",
                           ["2307.03172", "2404.06654"], task)


@tool
def research_retrieval(task: str) -> str:
    """Summarize how RETRIEVAL augmentation addresses long-context limits.

    Delegates to a specialist sub-agent that reads the retrieval paper(s) and
    returns a concise (~300 word) summary.

    Args:
        task: The overall comparison task, for context.
    """
    return _run_specialist("retrieval", "retrieval-augmented approaches",
                           ["2005.11401"], task)


print("3 specialist tools defined.")

3 specialist tools defined.


### Step 2 — Build the orchestrator

A regular Strands `Agent` with the 3 specialist tools.
It does NOT load papers — it reads the task, calls specialists, then synthesizes.
The tool docstrings are what the model reads to decide which tool does what.

In [ ]:
_subagent_usage.clear()   # reset before this run

ORCHESTRATOR_SYSTEM = (
    "You are a research coordinator with three independent specialist tools: "
    "research_attention, research_position, research_retrieval. "
    "Call ALL THREE tools to gather specialist summaries, then synthesize them "
    "into a single structured comparison."
)

orchestrator = Agent(
    model=_build_model(SONNET_MODEL, max_tokens=2000),  # needs room to synthesize 3 summaries
    tools=[research_attention, research_position, research_retrieval],
    system_prompt=ORCHESTRATOR_SYSTEM,  
    callback_handler=None,
)
print("Orchestrator built.")

Orchestrator built.


In [12]:
orchestrator.messages

[{'role': 'user',
  'content': [{'text': 'Compare how three areas handle long-context problems: (1) attention/architecture, (2) position effects, (3) retrieval. Cover the relevant paper(s) for each area.'}]},
 {'role': 'assistant',
  'content': [{'text': "I'll kick off all three specialist research agents **simultaneously** since they are fully independent of one another. This way we get all three summaries in parallel before synthesizing."},
   {'toolUse': {'toolUseId': 'tooluse_VAqQ8cJe6jJia6Sq8wx65a',
     'name': 'research_attention',
     'input': {'task': 'Compare how three areas handle long-context problems: (1) attention/architecture, (2) position effects, (3) retrieval. Cover the relevant paper(s) for each area.'}}},
   {'toolUse': {'toolUseId': 'tooluse_8BIad4Oy46AgHgJCzZvArX',
     'name': 'research_position',
     'input': {'task': 'Compare how three areas handle long-context problems: (1) attention/architecture, (2) position effects, (3) retrieval. Cover the relevant paper

In [7]:
t0 = time.time()
multi_result  = orchestrator(TASK)
multi_latency = time.time() - t0

multi_answer = _result_text(multi_result)
multi_peak   = peak_input_tokens(multi_result)
print("Multi-agent run done.")


Multi-agent run done.


### Peek — how Sonnet called the tools

This shows whether the orchestrator batched all 3 calls in ONE turn (→ parallel)
or called them one at a time (→ sequential).


In [8]:
print("=== Orchestrator message breakdown ===")
for i, msg in enumerate(orchestrator.messages):
    role    = msg["role"]
    content = msg.get("content", [])
    uses    = [b for b in content if isinstance(b, dict) and "toolUse"    in b]
    results = [b for b in content if isinstance(b, dict) and "toolResult" in b]
    if uses:
        names = [b["toolUse"]["name"] for b in uses]
        print(f"  Turn {i} [{role}]: {len(uses)} tool call(s) → {names}")
    elif results:
        print(f"  Turn {i} [{role}]: {len(results)} tool result(s) returned")
    else:
        chars = sum(len(b.get("text", "")) for b in content if isinstance(b, dict))
        print(f"  Turn {i} [{role}]: text ({chars} chars)")

max_per_turn = max(
    sum(1 for b in msg.get("content", []) if isinstance(b, dict) and "toolUse" in b)
    for msg in orchestrator.messages if msg["role"] == "assistant"
)
print()
print(f"Max tool calls in one turn: {max_per_turn}")
print("✅ Parallel via ConcurrentToolExecutor" if max_per_turn == 3 else "⚠️  Sequential")


=== Orchestrator message breakdown ===
  Turn 0 [user]: text (161 chars)
  Turn 1 [assistant]: 3 tool call(s) → ['research_attention', 'research_position', 'research_retrieval']
  Turn 2 [user]: 3 tool result(s) returned
  Turn 3 [assistant]: text (6294 chars)

Max tool calls in one turn: 3
✅ Parallel via ConcurrentToolExecutor


In [9]:
print("=== Sub-agent usage (each ran in its own window) ===")
for u in _subagent_usage:
    print(f"  {u['name']:12s}: {u['peak_tokens']:>6,} tokens  papers={u['papers']}")
sub_total = sum(u["peak_tokens"] for u in _subagent_usage)
print()
print(f"Sub-agent tokens (hidden from orchestrator): {sub_total:,}")
print(f"Orchestrator peak (summaries only):          {multi_peak:,}")


=== Sub-agent usage (each ran in its own window) ===
  retrieval   : 22,162 tokens  papers=['2005.11401']
  attention   : 12,161 tokens  papers=['1706.03762']
  position    : 47,529 tokens  papers=['2307.03172', '2404.06654']

Sub-agent tokens (hidden from orchestrator): 81,852
Orchestrator peak (summaries only):          2,961


In [10]:
# Timing proof: did all 3 specialists run at the same time?
# If parallel: start times are clustered; windows OVERLAP.
# If sequential: each specialist starts only after the previous one finishes.

t_base = min(u["t_start"] for u in _subagent_usage)  # earliest start = t=0

print("=== Parallel execution proof (wall-clock timing) ===")
print()
scale = 2  # chars per second
for u in sorted(_subagent_usage, key=lambda u: u["t_start"]):
    s = u["t_start"] - t_base
    e = u["t_end"]   - t_base
    bar = " " * int(s * scale) + "█" * max(1, int((e - s) * scale))
    print(f"  {u['name']:12s} start={s:5.1f}s  end={e:5.1f}s  thread={u['thread']}")
    print(f"  {'':12s} |{bar}|")

print()
all_starts = [u["t_start"] - t_base for u in _subagent_usage]
total_span = max(u["t_end"] for u in _subagent_usage) - t_base
sequential_sum = sum(u["t_end"] - u["t_start"] for u in _subagent_usage)

print(f"All 3 started within {max(all_starts):.1f}s of each other")
print(f"Total wall-clock elapsed:  {total_span:.1f}s")
print(f"Sum if sequential would be: {sequential_sum:.1f}s")
if total_span < sequential_sum * 0.7:
    print("✅ Parallel: elapsed ≈ slowest specialist, not sum of all three")

=== Parallel execution proof (wall-clock timing) ===

  position     start=  0.0s  end= 24.9s  thread=asyncio_2
               |█████████████████████████████████████████████████|
  retrieval    start=  0.0s  end= 16.4s  thread=asyncio_0
               |████████████████████████████████|
  attention    start=  0.0s  end= 16.8s  thread=asyncio_1
               |█████████████████████████████████|

All 3 started within 0.0s of each other
Total wall-clock elapsed:  24.9s
Sum if sequential would be: 58.1s
✅ Parallel: elapsed ≈ slowest specialist, not sum of all three


---
## Comparison


In [11]:
ratio = single_peak / max(multi_peak, 1)

print(f"{'Metric':<35} {'Single':>10} {'Multi':>10}")
print(f"{'──────':<35} {'──────':>10} {'─────':>10}")
print(f"{'Peak context (parent window)  tokens':<35} {single_peak:>10,} {multi_peak:>10,}")
print(f"{'Compression ratio':<35} {'1×':>10} {ratio:>9.0f}×")
print(f"{'Latency  seconds':<35} {single_latency:>9.1f}s {multi_latency:>9.1f}s")
print()
print("Sub-agent work (never seen by the orchestrator):")
for u in _subagent_usage:
    print(f"  {u['name']:12s}: {u['peak_tokens']:,} tokens")


Metric                                  Single      Multi
──────                                  ──────      ─────
Peak context (parent window)  tokens     80,059      2,961
Compression ratio                           1×        27×
Latency  seconds                         38.4s      57.6s

Sub-agent work (never seen by the orchestrator):
  retrieval   : 22,162 tokens
  attention   : 12,161 tokens
  position    : 47,529 tokens


---
## What this proves

| Observation | Why it matters |
|---|---|
| Orchestrator peak << single agent peak | The parent never sees raw papers — only summaries |
| 3 tool calls in one turn | Sonnet 4.6 recognises independent tools and batches them |
| Sub-agent tokens hidden from parent | Isolation is architectural, not configured |
| Latency ≈ slowest specialist | ConcurrentToolExecutor gives parallel execution for free |

**The key insight for context engineering:**
The boundary between the orchestrator and its specialist tools IS the compression
mechanism. You don't need a special algorithm — just stop letting raw exploration
enter the parent's context window.

**Scaling:** adding a new research area requires only:
1. A paper with a `field` tag in `data/index.json`
2. A new `@tool` specialist function

No changes to the orchestrator or routing logic needed.
